# Pretrained Keyword Spotting Model - TensorFlow Lite for Microcontrollers

This notebook demonstrates how to use a **pre-trained 20 kB Simple Audio Recognition model** from TensorFlow Lite for Microcontrollers to recognize keywords.

## 🎯 Learning Objectives

1. Download and setup the pretrained TensorFlow micro_speech model
2. Understand model architecture and configuration parameters
3. Convert models from TensorFlow to TensorFlow Lite format
4. Apply quantization to reduce model size (float → int8)
5. Evaluate accuracy on test datasets
6. Test with custom audio recordings

## �� What You'll Build

- **Float Model**: ~84 KB (baseline accuracy)
- **Quantized Model**: ~20 KB (4x smaller, minimal accuracy loss)
- **Target Keywords**: "yes" and "no"
- **Deployment Ready**: Optimized for microcontrollers

## Step 1: Download TensorFlow Source Code

In [ ]:
# Download TensorFlow v2.14.0 source code
!wget https://github.com/tensorflow/tensorflow/archive/v2.14.0.zip
!unzip v2.14.0.zip &> 0
!mv tensorflow-2.14.0/ tensorflow/

## Step 2: Import Libraries

In [ ]:
import tensorflow.compat.v1 as tf
import sys
import numpy as np
import pickle

# Add TensorFlow speech commands utilities to path
sys.path.append("/content/tensorflow/tensorflow/examples/speech_commands/")
import input_data
import models

## Step 3: Configure Keywords

In [ ]:
# Define target keywords
WANTED_WORDS = "yes,no"
print("Spotting these words: %s" % WANTED_WORDS)

# Calculate balanced class distribution
number_of_labels = WANTED_WORDS.count(',') + 1
number_of_total_labels = number_of_labels + 2
equal_percentage_of_training_samples = int(100.0/(number_of_total_labels))
SILENT_PERCENTAGE = equal_percentage_of_training_samples
UNKNOWN_PERCENTAGE = equal_percentage_of_training_samples

In [ ]:
# Constants shared during training and inference
PREPROCESS = 'micro'
WINDOW_STRIDE = 20
MODEL_ARCHITECTURE = 'tiny_conv'

# Training directories
DATASET_DIR = 'dataset/'
LOGS_DIR = 'logs/'
TRAIN_DIR = 'train/'

# Inference directories
import os
MODELS_DIR = 'models'
if not os.path.exists(MODELS_DIR):
    os.mkdir(MODELS_DIR)
MODEL_TF = os.path.join(MODELS_DIR, 'model.pb')
MODEL_TFLITE = os.path.join(MODELS_DIR, 'model.tflite')
FLOAT_MODEL_TFLITE = os.path.join(MODELS_DIR, 'float_model.tflite')
MODEL_TFLITE_MICRO = os.path.join(MODELS_DIR, 'model.cc')
SAVED_MODEL = os.path.join(MODELS_DIR, 'saved_model')

# Quantization constants
QUANT_INPUT_MIN = 0.0
QUANT_INPUT_MAX = 26.0
QUANT_INPUT_RANGE = QUANT_INPUT_MAX - QUANT_INPUT_MIN

# Audio processing parameters
SAMPLE_RATE = 16000
CLIP_DURATION_MS = 1000
WINDOW_SIZE_MS = 30.0
FEATURE_BIN_COUNT = 40
BACKGROUND_FREQUENCY = 0.8
BACKGROUND_VOLUME_RANGE = 0.1
TIME_SHIFT_MS = 100.0

# Dataset configuration
DATA_URL = 'https://storage.googleapis.com/download.tensorflow.org/data/speech_commands_v0.02.tar.gz'
VALIDATION_PERCENTAGE = 10
TESTING_PERCENTAGE = 10

## Step 4: Download Pretrained Checkpoint

In [ ]:
!curl -O "https://storage.googleapis.com/download.tensorflow.org/models/tflite/speech_micro_train_2020_05_10.tgz"
!tar xzf speech_micro_train_2020_05_10.tgz
TOTAL_STEPS = 15000

## Step 5: Convert to SavedModel

In [ ]:
!rm -rf {SAVED_MODEL}
!python tensorflow/tensorflow/examples/speech_commands/freeze.py \
--wanted_words=$WANTED_WORDS \
--window_stride_ms=$WINDOW_STRIDE \
--preprocess=$PREPROCESS \
--model_architecture=$MODEL_ARCHITECTURE \
--start_checkpoint=$TRAIN_DIR$MODEL_ARCHITECTURE'.ckpt-'{TOTAL_STEPS} \
--save_format=saved_model \
--output_file={SAVED_MODEL}

## Step 6: Setup Audio Processor

In [ ]:
model_settings = models.prepare_model_settings(
    len(input_data.prepare_words_list(WANTED_WORDS.split(','))),
    SAMPLE_RATE, CLIP_DURATION_MS, WINDOW_SIZE_MS,
    WINDOW_STRIDE, FEATURE_BIN_COUNT, PREPROCESS)

audio_processor = input_data.AudioProcessor(
    DATA_URL, DATASET_DIR,
    SILENT_PERCENTAGE, UNKNOWN_PERCENTAGE,
    WANTED_WORDS.split(','), VALIDATION_PERCENTAGE,
    TESTING_PERCENTAGE, model_settings, LOGS_DIR)

## Step 7: Convert to TFLite (Float & Quantized)

In [ ]:
with tf.Session() as sess:
    # Float model conversion
    float_converter = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL)
    float_tflite_model = float_converter.convert()
    float_tflite_model_size = open(FLOAT_MODEL_TFLITE, "wb").write(float_tflite_model)
    print("Float model is %d bytes" % float_tflite_model_size)

    # Quantized model conversion
    converter = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.inference_input_type = tf.lite.constants.INT8
    converter.inference_output_type = tf.lite.constants.INT8
    
    def representative_dataset_gen():
        for i in range(100):
            data, _ = audio_processor.get_data(1, i*1, model_settings,
                                             BACKGROUND_FREQUENCY, 
                                             BACKGROUND_VOLUME_RANGE,
                                             TIME_SHIFT_MS,
                                             'testing',
                                             sess)
            flattened_data = np.array(data.flatten(), dtype=np.float32).reshape(1, 1960)
            yield [flattened_data]
    
    converter.representative_dataset = representative_dataset_gen
    tflite_model = converter.convert()
    tflite_model_size = open(MODEL_TFLITE, "wb").write(tflite_model)
    print("Quantized model is %d bytes" % tflite_model_size)

## Step 8: Evaluate Models

In [ ]:
def run_tflite_inference_testSet(tflite_model_path, model_type="Float"):
    np.random.seed(0)
    with tf.Session() as sess:
        test_data, test_labels = audio_processor.get_data(
            -1, 0, model_settings, BACKGROUND_FREQUENCY, BACKGROUND_VOLUME_RANGE,
            TIME_SHIFT_MS, 'testing', sess)
    test_data = np.expand_dims(test_data, axis=1).astype(np.float32)

    interpreter = tf.lite.Interpreter(tflite_model_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]
    
    if model_type == "Quantized":
        input_scale, input_zero_point = input_details["quantization"]
        test_data = test_data / input_scale + input_zero_point
        test_data = test_data.astype(input_details["dtype"])

    correct_predictions = 0
    for i in range(len(test_data)):
        interpreter.set_tensor(input_details["index"], test_data[i])
        interpreter.invoke()
        output = interpreter.get_tensor(output_details["index"])[0]
        top_prediction = output.argmax()
        correct_predictions += (top_prediction == test_labels[i])

    print('%s model accuracy is %f%% (Number of test samples=%d)' % (
        model_type, (correct_predictions * 100) / len(test_data), len(test_data)))

# Evaluate models
run_tflite_inference_testSet(FLOAT_MODEL_TFLITE)
run_tflite_inference_testSet(MODEL_TFLITE, model_type='Quantized')

## Step 9: Download & Load Sample Audio

In [ ]:
from IPython.display import HTML, Audio
!wget --no-check-certificate --content-disposition https://github.com/tinyMLx/colabs/blob/master/yes_no.pkl?raw=true

fid = open('yes_no.pkl', 'rb')
audio_files = pickle.load(fid)
yes1 = audio_files['yes1']
yes2 = audio_files['yes2']
yes3 = audio_files['yes3']
yes4 = audio_files['yes4']
no1 = audio_files['no1']
no2 = audio_files['no2']
no3 = audio_files['no3']
no4 = audio_files['no4']
sr_yes1 = audio_files['sr_yes1']
sr_yes2 = audio_files['sr_yes2']
sr_yes3 = audio_files['sr_yes3']
sr_yes4 = audio_files['sr_yes4']
sr_no1 = audio_files['sr_no1']
sr_no2 = audio_files['sr_no2']
sr_no3 = audio_files['sr_no3']
sr_no4 = audio_files['sr_no4']

## Step 10: Install Audio Tools

In [ ]:
!pip install ffmpeg-python &> 0
from google.colab.output import eval_js
from base64 import b64decode
import numpy as np
from scipy.io.wavfile import read as wav_read
import io
import ffmpeg
!pip install librosa
import librosa
import scipy.io.wavfile
!git clone https://github.com/petewarden/extract_loudest_section.git
!make -C extract_loudest_section/
print("Packages Installed, Extract_Loudest_Section Built")

## Step 11: Define Single-File Inference

In [ ]:
TF_SESS = tf.compat.v1.InteractiveSession()

def run_tflite_inference_singleFile(tflite_model_path, custom_audio, sr_custom_audio, model_type="Float"):
    # Resample to 16kHz
    custom_audio_resampled = librosa.resample(librosa.to_mono(np.float64(custom_audio)), 
                                               orig_sr=sr_custom_audio, target_sr=SAMPLE_RATE)
    # Extract loudest 1 second
    scipy.io.wavfile.write('custom_audio.wav', SAMPLE_RATE, np.int16(custom_audio_resampled))
    !/tmp/extract_loudest_section/gen/bin/extract_loudest_section custom_audio.wav ./trimmed
    
    # Generate features
    custom_model_settings = models.prepare_model_settings(
        0, SAMPLE_RATE, CLIP_DURATION_MS, WINDOW_SIZE_MS,
        WINDOW_STRIDE, FEATURE_BIN_COUNT, PREPROCESS)
    custom_audio_processor = input_data.AudioProcessor(None, None, 0, 0, '', 0, 0,
                                                      model_settings, None)
    custom_audio_preprocessed = custom_audio_processor.get_features_for_wav(
                                          'trimmed/custom_audio.wav', model_settings, TF_SESS)
    custom_audio_input = custom_audio_preprocessed[0].flatten()
    test_data = np.reshape(custom_audio_input,(1,len(custom_audio_input)))

    # Initialize interpreter
    interpreter = tf.lite.Interpreter(tflite_model_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    # Quantize if needed
    if model_type == "Quantized":
        input_scale, input_zero_point = input_details["quantization"]
        test_data = test_data / input_scale + input_zero_point
        test_data = test_data.astype(input_details["dtype"])

    # Run inference
    interpreter.set_tensor(input_details["index"], test_data)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details["index"])[0]
    top_prediction = output.argmax()

    # Decode prediction
    if top_prediction == 2 or top_prediction == 3:
        top_prediction_str = WANTED_WORDS.split(',')[top_prediction-2]
    elif top_prediction == 0:
        top_prediction_str = 'silence'
    else:
        top_prediction_str = 'unknown'

    print('%s model guessed the value to be %s' % (model_type, top_prediction_str))

## Step 12: Test on Sample Files

In [ ]:
print("Testing yes1")
run_tflite_inference_singleFile(MODEL_TFLITE, yes1, sr_yes1, model_type="Quantized")
print("Testing yes2")
run_tflite_inference_singleFile(MODEL_TFLITE, yes2, sr_yes2, model_type="Quantized")
print("Testing yes3")
run_tflite_inference_singleFile(MODEL_TFLITE, yes3, sr_yes3, model_type="Quantized")
print("Testing yes4")
run_tflite_inference_singleFile(MODEL_TFLITE, yes4, sr_yes4, model_type="Quantized")
print("Testing no1")
run_tflite_inference_singleFile(MODEL_TFLITE, no1, sr_no1, model_type="Quantized")
print("Testing no2")
run_tflite_inference_singleFile(MODEL_TFLITE, no2, sr_no2, model_type="Quantized")
print("Testing no3")
run_tflite_inference_singleFile(MODEL_TFLITE, no3, sr_no3, model_type="Quantized")
print("Testing no4")
run_tflite_inference_singleFile(MODEL_TFLITE, no4, sr_no4, model_type="Quantized")

## Step 13: Record Your Own Audio

In [ ]:
AUDIO_HTML = """
<script>
var my_div = document.createElement("DIV");
var my_btn = document.createElement("BUTTON");
my_btn.appendChild(document.createTextNode("Press to start recording"));
my_div.appendChild(my_btn);
document.body.appendChild(my_div);

var base64data = 0, reader, recorder, gumStream;
var recordButton = my_btn;

var handleSuccess = function(stream) {
  gumStream = stream;
  recorder = new MediaRecorder(stream, {bitsPerSecond: 128000, audioBitsPerSecond: 128000, mimeType: 'audio/mp4'});
  recorder.ondataavailable = function(e) {
    var url = URL.createObjectURL(e.data);
    var preview = document.createElement('audio');
    preview.controls = true;
    preview.src = url;
    document.body.appendChild(preview);
    reader = new FileReader();
    reader.readAsDataURL(e.data);
    reader.onloadend = function() { base64data = reader.result; }
  };
  recorder.start();
};

recordButton.innerText = "Recording... press to stop";
navigator.mediaDevices.getUserMedia({audio: true}).then(handleSuccess);

function toggleRecording() {
  if (recorder && recorder.state == "recording") {
      recorder.stop();
      gumStream.getAudioTracks()[0].stop();
      recordButton.innerText = "Saving the recording... pls wait!"
  }
}

function sleep(ms) { return new Promise(resolve => setTimeout(resolve, ms)); }

var data = new Promise(resolve=>{
  recordButton.onclick = ()=>{ toggleRecording(); sleep(2000).then(() => { resolve(base64data.toString()) }); }
});
</script>
"""

def get_audio():
    display(HTML(AUDIO_HTML))
    data = eval_js("data")
    binary = b64decode(data.split(',')[1])
    process = (ffmpeg.input('pipe:0').output('pipe:1', format='wav', ac='1')
               .run_async(pipe_stdin=True, pipe_stdout=True, pipe_stderr=True, 
                         quiet=True, overwrite_output=True))
    output, err = process.communicate(input=binary)
    riff_chunk_size = len(output) - 8
    q, b = riff_chunk_size, []
    for i in range(4):
        q, r = divmod(q, 256)
        b.append(r)
    riff = output[:4] + bytes(b) + output[8:]
    sr, audio = wav_read(io.BytesIO(riff))
    return audio, sr

print("Chrome Audio Recorder Defined")

In [ ]:
custom_audio, sr_custom_audio = get_audio()
print("DONE")

In [ ]:
run_tflite_inference_singleFile(MODEL_TFLITE, custom_audio, sr_custom_audio, model_type="Quantized")